# L5 — Decision Models: Structured, Thresholdable Decisions

## What you'll learn

- What a decision model is, and how it differs from asking a chat model for a label
- The three question types: yes or no, choice, and score
- How to turn a probability into an action with a threshold
- How to swap the decision provider by pasting a different cell

## Overview

A decision model reads text like a chat model, but instead of writing a reply it returns **answers with probabilities**: yes or no, one of your options, or a level on a scale. Your code then decides what to do with the numbers. Use it wherever you would otherwise ask a chat model for a label, such as flagging, classifying or grading.

You choose one of two providers in the **Partner Selection UI** (Application → Decision Models):

- **Strands Decider (AWS).** An open source model on a SageMaker endpoint in your account, set up by the workshop. You pay per instance-hour.
- **Perplexity Decisions.** A hosted service that needs an API key. You pay per input token.

Connecting a provider here also turns on triage in the chatbot. The orchestrator doesn't change.

## Architecture

![Application layer: the customer's message from the chatbot is triaged by a decision model (Strands Decider or Perplexity Decisions). Out-of-scope messages get a canned reply; in-scope messages go to the AgentCore Harness, which uses the Order, Refund and Research agents, and the reply streams back.](images/l5-application-architecture.png)

This notebook builds the Decision Model box at the front door of the diagram. The chatbot and the agent loop come from the other notebooks.

## Steps

1. Set up
2. Connect to your decision provider (code from the Partner Selection UI)
3. Try the three question types

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- The Partner Selection UI from L0 ([`1_partner_selection.ipynb`](../l0-partner-selection/1_partner_selection.ipynb)) is running.
- If you use Strands Decider, its SageMaker endpoint is ready. The workshop creates it for you, which usually takes about 20 minutes after the stack is created, and the UI shows its status.
- You can run this notebook in parallel with [`3_orchestrator_agent.ipynb`](../l3-orchestration/3_orchestrator_agent.ipynb).

## Step 1: Set up

Install the packages and load the helper code for the decision models.

In [ ]:
%pip install --quiet --upgrade boto3 requests

In [ ]:
import os
import importlib
import decision_models
importlib.reload(decision_models)  # pick up edits without restarting the kernel
from decision_models import (
    noul, choice, score, DecisionsError,
    URGENCY_LEVELS, SENTIMENT_OPTIONS, triage_message,
)

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")  # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"
print(f"Region: {REGION}. Helper loaded.")

## Step 2: Connect to your decision provider

Paste the **1 · Connect to the decision provider** code from the Partner Selection UI (Application → Decision Models) into the empty cell below, then run it. It connects to your provider and saves your choice, so the chatbot knows which one to use.

- **Strands Decider (AWS):** uses the endpoint the workshop created. If the endpoint is still starting, the cell waits, and it explains if the setup failed.
- **Perplexity Decisions:** reads your API key, saved in the UI, into memory.

To switch providers later, paste the other provider's code here and re-run from this cell down.

<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Application → Decision Models</b>.</li>
<li>Pick a decision provider. For Perplexity Decisions, save its API key.</li><li>Expand <b>📋 1 · Connect to the decision provider</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "1 · Connect to the decision provider" snippet from the Partner Selection UI here, then run.



In [ ]:
if "decider" not in globals():
    raise RuntimeError("No decision provider connected. Paste the Connect snippet from the "
                       "Partner Selection UI into the cell above and run it.")
print(f"Using: {decider.describe()}")

## Step 3: Try the three question types

Why not ask a chat model to "reply with one word: positive, neutral or negative"? It often gives the right word, but not how sure it is, and you still have to check the text it returns. A decision model returns answers with probabilities, so your code can say "flag this only when we are 80% sure".

Each sub-step below asks one kind of question about a few sample messages. The message is sent as data: a decision model doesn't follow instructions hidden in the content.

### 3.1 Yes or no: should a human agent take over?

A yes or no question returns one number: the probability that the answer is yes. You pick a threshold and compare. The orchestrator keeps answering either way; the flag tells your support team where to look.

In [ ]:
ESCALATE_QUESTION = noul(
    "Should a human support agent take over this conversation? Answer yes if the customer "
    "is very upset, threatens legal action or to cancel, reports a safety issue or possible "
    "fraud, or explicitly asks for a human."
)
messages = [
    "Can you tell me when my order ORD-10001 will arrive?",
    "This is the third time my refund failed. I want to talk to a real person right now.",
    "Someone placed an order on my account that I never made.",
]
for msg in messages:
    resp = decider.decide(state=msg, questions={"escalate": ESCALATE_QUESTION})
    p = resp["answers"]["escalate"]["noul"]
    verdict = "ESCALATE" if p >= 0.5 else "no escalation"
    print(f"p(yes)={p:0.3f}  [{verdict}]  ← {msg}")
latency = f", latency: {resp['latency_ms']} ms" if "latency_ms" in resp else ""
print(f"\n(input_tokens this call: {resp['usage']['input_tokens']}{latency})")

### 3.2 Choice: what is the customer's sentiment?

A choice question returns a probability for every option (they add up to about 1), the most likely option, and a confidence. Each option comes with a short description of when it applies.

In [ ]:
SENTIMENT_QUESTION = choice("What is the customer's sentiment in this message?", SENTIMENT_OPTIONS)
samples = [
    "Thanks so much, the replacement arrived early and works great!",
    "What's the return window for electronics?",
    "I've been waiting two weeks and nobody answers my emails.",
]
for msg in samples:
    resp = decider.decide(state=msg, questions={"sentiment": SENTIMENT_QUESTION})
    a = resp["answers"]["sentiment"]
    probs = "  ".join(f"{k}={v:0.2f}" for k, v in a["probabilities"].items())
    print(f"sentiment={a['choice']:<9} conf={a['confidence']:0.2f}   {probs}")
    print(f"    ← {msg}\n")

### 3.3 Score: how urgent is it?

A score question rates the message against ordered levels, from the lowest to the highest. The score is the probability-weighted average of the levels, so it can fall between two levels: 1.78 means mostly level 2, with some level 1. The chatbot grades urgency with the same levels.

In [ ]:
for level in URGENCY_LEVELS:
    print("  rubric:", level)
print()
tickets = [
    "Just wondering if you carry blue towels.",
    "My order is three days late and I need it for an event tomorrow.",
    "I was charged twice and the second charge is for an order I never placed.",
]
for t in tickets:
    resp = decider.decide(
        state=t,
        questions={"urgency": score("How urgent is this support ticket?", URGENCY_LEVELS)},
    )
    a = resp["answers"]["urgency"]
    print(f"score={a['score']:0.2f} (conf {a['confidence']:0.2f})   ← {t}")
print("\nlegend:", a["legend"])

## Summary

This notebook connected a decision provider, Strands Decider on a SageMaker endpoint or Perplexity Decisions, and asked it three kinds of question: yes or no, pick one option, and rate on a scale. Each answer came back as probabilities, which your code compares with a threshold to decide what to do.

**Next:** [`2_chatbot_ui.ipynb`](2_chatbot_ui.ipynb). Launch the chatbot, which uses this decision step to triage every customer message.

---
## Cleanup

There is nothing to delete in this notebook.

- **Strands Decider:** the endpoint belongs to the workshop stack and is deleted with it. It bills every hour until then.
- **Perplexity Decisions:** the key was held in memory only and is discarded when the kernel stops. It stays in SSM until you delete it, and the same key is used by the web-search tool in L3.